In [1]:
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import src.train_utils as T
import itertools
import random

In [2]:
df = pd.read_csv('../datasets/dataset_base.csv')
df['date'] = pd.to_datetime(df['date'])

train_df = df[df['date'].dt.year < 2022]

In [3]:
train_df.head()

,row,col,date,pm25_t,u_wind_10m,v_wind_10m,dew_temp_2m,temp_2m,surf_pressure,precip_sum,frp,elevation,delta_pm25_t+1,delta_pm25_t
0,0,0,2018-07-05,8.317946,1.432791,0.080936,291.41534,295.22916,85304.766,0.003423,0.0,1360.0,0.601407,0.302968
1,0,0,2018-07-06,8.919353,0.748148,0.110430,291.74884,295.01360,85287.414,0.007418,0.0,1360.0,0.038002,0.601407
2,0,0,2018-07-07,8.957355,0.922457,0.196369,292.11148,294.27896,85314.555,0.013237,0.0,1360.0,0.556278,0.038002
3,0,0,2018-07-08,9.513633,1.040234,0.047216,291.94238,293.10020,85344.050,0.011159,0.0,1360.0,-0.491020,0.556278
4,0,0,2018-07-09,9.022613,1.154719,0.083273,290.86307,293.92580,85426.945,0.001921,0.0,1360.0,1.041161,-0.491020


In [4]:
control_df = train_df[['row', 'col', 'date', 'delta_pm25_t', 'delta_pm25_t+1']]
control_df.head()

,row,col,date,delta_pm25_t,delta_pm25_t+1
0,0,0,2018-07-05,0.302968,0.601407
1,0,0,2018-07-06,0.601407,0.038002
2,0,0,2018-07-07,0.038002,0.556278
3,0,0,2018-07-08,0.556278,-0.491020
4,0,0,2018-07-09,-0.491020,1.041161


In [5]:
params = {'max_depth': 4, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'n_estimators': 500}

## + PM2.5

In [7]:
pm25_df = train_df[['row', 'col', 'date', 'pm25_t', 'delta_pm25_t', 'delta_pm25_t+1']]
pm25_df.head()

,row,col,date,pm25_t,delta_pm25_t,delta_pm25_t+1
0,0,0,2018-07-05,8.317946,0.302968,0.601407
1,0,0,2018-07-06,8.919353,0.601407,0.038002
2,0,0,2018-07-07,8.957355,0.038002,0.556278
3,0,0,2018-07-08,9.513633,0.556278,-0.491020
4,0,0,2018-07-09,9.022613,-0.491020,1.041161


In [8]:
model = xgb.XGBRegressor(
        objective='reg:squarederror',
        **params,
        random_state=191
    )

rmse_scores, mae_scores = T.rolling_window_cv(
    train_days=365 * 2,
    gap_days=21,
    test_days=60,
    df=control_df,
    model=model
)

mean_rmse = np.mean(rmse_scores)
se_rmse = np.std(rmse_scores)

mean_mae = np.mean(mae_scores)
se_mae = np.std(mae_scores)

print("Summary of Performance Statistics")
print(f"RMSE: {mean_rmse} +/- {se_rmse}")
print(f"MAE: {mean_mae} +/- {se_mae}")

Fold 1
2018-07-05 00:00:00 2020-07-19 00:00:00
2020-08-10 00:00:00 2020-10-08 00:00:00
RMSE: 1.6608790479631468
MAE: 1.0045632047437956
Fold 2
2018-09-03 00:00:00 2020-09-17 00:00:00
2020-10-09 00:00:00 2020-12-07 00:00:00
RMSE: 5.264362787323237
MAE: 2.177423560437211
Fold 3
2018-11-02 00:00:00 2020-11-16 00:00:00
2020-12-08 00:00:00 2021-02-05 00:00:00
RMSE: 7.309145973356997
MAE: 5.278711184027159
Fold 4
2019-01-01 00:00:00 2021-01-15 00:00:00
2021-02-06 00:00:00 2021-04-18 00:00:00
RMSE: 13.47693459207647
MAE: 9.305156073644321
Fold 5
2019-03-04 00:00:00 2021-03-28 00:00:00
2021-04-19 00:00:00 2021-06-17 00:00:00
RMSE: 4.84874163426322
MAE: 2.6851162869278324
Fold 6
2019-05-03 00:00:00 2021-05-27 00:00:00
2021-06-18 00:00:00 2021-08-16 00:00:00
RMSE: 1.052364904871975
MAE: 0.8183504835756414
Fold 7
2019-07-02 00:00:00 2021-07-26 00:00:00
2021-08-17 00:00:00 2021-10-15 00:00:00
RMSE: 1.2840076910823286
MAE: 0.9222275354104852
Fold 8
2019-08-31 00:00:00 2021-09-24 00:00:00
2021-10-16

## + PM2.5

In [13]:
wind_df = train_df[['row', 'col', 'date', 'u_wind_10m', 'v_wind_10m', 'delta_pm25_t', 'delta_pm25_t+1']]
wind_df.head()

,row,col,date,u_wind_10m,v_wind_10m,delta_pm25_t,delta_pm25_t+1
0,0,0,2018-07-05,1.432791,0.080936,0.302968,0.601407
1,0,0,2018-07-06,0.748148,0.110430,0.601407,0.038002
2,0,0,2018-07-07,0.922457,0.196369,0.038002,0.556278
3,0,0,2018-07-08,1.040234,0.047216,0.556278,-0.491020
4,0,0,2018-07-09,1.154719,0.083273,-0.491020,1.041161


In [14]:
model = xgb.XGBRegressor(
        objective='reg:squarederror',
        **params,
        random_state=191
    )

rmse_scores, mae_scores = T.rolling_window_cv(
    train_days=365 * 2,
    gap_days=21,
    test_days=60,
    df=wind_df,
    model=model
)

mean_rmse = np.mean(rmse_scores)
se_rmse = np.std(rmse_scores)

mean_mae = np.mean(mae_scores)
se_mae = np.std(mae_scores)

print("Summary of Performance Statistics")
print(f"RMSE: {mean_rmse} +/- {se_rmse}")
print(f"MAE: {mean_mae} +/- {se_mae}")

Fold 1
2018-07-05 00:00:00 2020-07-19 00:00:00
2020-08-10 00:00:00 2020-10-08 00:00:00
RMSE: 1.7486801166627806
MAE: 1.0563562520894316
Fold 2
2018-09-03 00:00:00 2020-09-17 00:00:00
2020-10-09 00:00:00 2020-12-07 00:00:00
RMSE: 5.240916635713008
MAE: 2.2501243263484723
Fold 3
2018-11-02 00:00:00 2020-11-16 00:00:00
2020-12-08 00:00:00 2021-02-05 00:00:00
RMSE: 7.327224849883075
MAE: 5.301939492957378
Fold 4
2019-01-01 00:00:00 2021-01-15 00:00:00
2021-02-06 00:00:00 2021-04-18 00:00:00
RMSE: 13.882472075859773
MAE: 9.485853289578783
Fold 5
2019-03-04 00:00:00 2021-03-28 00:00:00
2021-04-19 00:00:00 2021-06-17 00:00:00
RMSE: 4.898510072367062
MAE: 2.7328839985788878
Fold 6
2019-05-03 00:00:00 2021-05-27 00:00:00
2021-06-18 00:00:00 2021-08-16 00:00:00
RMSE: 1.0935588009169859
MAE: 0.8361755654996341
Fold 7
2019-07-02 00:00:00 2021-07-26 00:00:00
2021-08-17 00:00:00 2021-10-15 00:00:00
RMSE: 1.3814117715128464
MAE: 1.0003617648737775
Fold 8
2019-08-31 00:00:00 2021-09-24 00:00:00
2021-1

## + Wind Speed

In [19]:
wind_df_2 = wind_df.copy()

wind_df_2['wind_speed'] = np.sqrt(wind_df_2['u_wind_10m'] ** 2 + wind_df_2['v_wind_10m'] ** 2)

wind_df_2 = wind_df_2.drop(['u_wind_10m', 'v_wind_10m'], axis=1)
wind_df_2.head()

,row,col,date,delta_pm25_t,delta_pm25_t+1,wind_speed
0,0,0,2018-07-05,0.302968,0.601407,1.435076
1,0,0,2018-07-06,0.601407,0.038002,0.756254
2,0,0,2018-07-07,0.038002,0.556278,0.943126
3,0,0,2018-07-08,0.556278,-0.491020,1.041305
4,0,0,2018-07-09,-0.491020,1.041161,1.157717


In [20]:
model = xgb.XGBRegressor(
        objective='reg:squarederror',
        **params,
        random_state=191
    )

rmse_scores, mae_scores = T.rolling_window_cv(
    train_days=365 * 2,
    gap_days=21,
    test_days=60,
    df=wind_df_2,
    model=model
)

mean_rmse = np.mean(rmse_scores)
se_rmse = np.std(rmse_scores)

mean_mae = np.mean(mae_scores)
se_mae = np.std(mae_scores)

print("Summary of Performance Statistics")
print(f"RMSE: {mean_rmse} +/- {se_rmse}")
print(f"MAE: {mean_mae} +/- {se_mae}")

Fold 1
2018-07-05 00:00:00 2020-07-19 00:00:00
2020-08-10 00:00:00 2020-10-08 00:00:00
RMSE: 1.6821011319225758
MAE: 1.0489153260050632
Fold 2
2018-09-03 00:00:00 2020-09-17 00:00:00
2020-10-09 00:00:00 2020-12-07 00:00:00
RMSE: 5.2821053748538285
MAE: 2.215849848255138
Fold 3
2018-11-02 00:00:00 2020-11-16 00:00:00
2020-12-08 00:00:00 2021-02-05 00:00:00
RMSE: 7.295524867683109
MAE: 5.290948588111716
Fold 4
2019-01-01 00:00:00 2021-01-15 00:00:00
2021-02-06 00:00:00 2021-04-18 00:00:00
RMSE: 13.494441069871492
MAE: 9.306498814266893
Fold 5
2019-03-04 00:00:00 2021-03-28 00:00:00
2021-04-19 00:00:00 2021-06-17 00:00:00
RMSE: 4.885578309594897
MAE: 2.7305554529680682
Fold 6
2019-05-03 00:00:00 2021-05-27 00:00:00
2021-06-18 00:00:00 2021-08-16 00:00:00
RMSE: 1.0689301001951947
MAE: 0.8269320219324455
Fold 7
2019-07-02 00:00:00 2021-07-26 00:00:00
2021-08-17 00:00:00 2021-10-15 00:00:00
RMSE: 1.31018905906414
MAE: 0.9541729091191229
Fold 8
2019-08-31 00:00:00 2021-09-24 00:00:00
2021-10-

## + Temperature

In [21]:
temp_df = train_df[['row', 'col', 'date', 'temp_2m', 'delta_pm25_t', 'delta_pm25_t+1']]
temp_df.head()

,row,col,date,temp_2m,delta_pm25_t,delta_pm25_t+1
0,0,0,2018-07-05,295.22916,0.302968,0.601407
1,0,0,2018-07-06,295.01360,0.601407,0.038002
2,0,0,2018-07-07,294.27896,0.038002,0.556278
3,0,0,2018-07-08,293.10020,0.556278,-0.491020
4,0,0,2018-07-09,293.92580,-0.491020,1.041161


In [22]:
model = xgb.XGBRegressor(
        objective='reg:squarederror',
        **params,
        random_state=191
    )

rmse_scores, mae_scores = T.rolling_window_cv(
    train_days=365 * 2,
    gap_days=21,
    test_days=60,
    df=temp_df,
    model=model
)

mean_rmse = np.mean(rmse_scores)
se_rmse = np.std(rmse_scores)

mean_mae = np.mean(mae_scores)
se_mae = np.std(mae_scores)

print("Summary of Performance Statistics")
print(f"RMSE: {mean_rmse} +/- {se_rmse}")
print(f"MAE: {mean_mae} +/- {se_mae}")

Fold 1
2018-07-05 00:00:00 2020-07-19 00:00:00
2020-08-10 00:00:00 2020-10-08 00:00:00
RMSE: 1.693161993269757
MAE: 1.046662534373758
Fold 2
2018-09-03 00:00:00 2020-09-17 00:00:00
2020-10-09 00:00:00 2020-12-07 00:00:00
RMSE: 5.268949918763356
MAE: 2.196257252124272
Fold 3
2018-11-02 00:00:00 2020-11-16 00:00:00
2020-12-08 00:00:00 2021-02-05 00:00:00
RMSE: 7.280476347403224
MAE: 5.248203819876261
Fold 4
2019-01-01 00:00:00 2021-01-15 00:00:00
2021-02-06 00:00:00 2021-04-18 00:00:00
RMSE: 13.463969401651683
MAE: 9.300208181947898
Fold 5
2019-03-04 00:00:00 2021-03-28 00:00:00
2021-04-19 00:00:00 2021-06-17 00:00:00
RMSE: 4.886385686969198
MAE: 2.74700246575093
Fold 6
2019-05-03 00:00:00 2021-05-27 00:00:00
2021-06-18 00:00:00 2021-08-16 00:00:00
RMSE: 1.090273046190975
MAE: 0.8554486926592115
Fold 7
2019-07-02 00:00:00 2021-07-26 00:00:00
2021-08-17 00:00:00 2021-10-15 00:00:00
RMSE: 1.3086589773926223
MAE: 0.9467741255661004
Fold 8
2019-08-31 00:00:00 2021-09-24 00:00:00
2021-10-16 0

## + Precipitation

In [23]:
precip_df = train_df[['row', 'col', 'date', 'precip_sum', 'delta_pm25_t', 'delta_pm25_t+1']]

precip_df.head()

,row,col,date,precip_sum,delta_pm25_t,delta_pm25_t+1
0,0,0,2018-07-05,0.003423,0.302968,0.601407
1,0,0,2018-07-06,0.007418,0.601407,0.038002
2,0,0,2018-07-07,0.013237,0.038002,0.556278
3,0,0,2018-07-08,0.011159,0.556278,-0.491020
4,0,0,2018-07-09,0.001921,-0.491020,1.041161


In [24]:
model = xgb.XGBRegressor(
        objective='reg:squarederror',
        **params,
        random_state=191
    )

rmse_scores, mae_scores = T.rolling_window_cv(
    train_days=365 * 2,
    gap_days=21,
    test_days=60,
    df=precip_df,
    model=model
)

mean_rmse = np.mean(rmse_scores)
se_rmse = np.std(rmse_scores)

mean_mae = np.mean(mae_scores)
se_mae = np.std(mae_scores)

print("Summary of Performance Statistics")
print(f"RMSE: {mean_rmse} +/- {se_rmse}")
print(f"MAE: {mean_mae} +/- {se_mae}")

Fold 1
2018-07-05 00:00:00 2020-07-19 00:00:00
2020-08-10 00:00:00 2020-10-08 00:00:00
RMSE: 1.6699406799927803
MAE: 0.9851963545365923
Fold 2
2018-09-03 00:00:00 2020-09-17 00:00:00
2020-10-09 00:00:00 2020-12-07 00:00:00
RMSE: 6.032526229727899
MAE: 3.002481314137816
Fold 3
2018-11-02 00:00:00 2020-11-16 00:00:00
2020-12-08 00:00:00 2021-02-05 00:00:00
RMSE: 7.881191154824163
MAE: 5.978189079510806
Fold 4
2019-01-01 00:00:00 2021-01-15 00:00:00
2021-02-06 00:00:00 2021-04-18 00:00:00
RMSE: 13.515672284567215
MAE: 9.394272543034415
Fold 5
2019-03-04 00:00:00 2021-03-28 00:00:00
2021-04-19 00:00:00 2021-06-17 00:00:00
RMSE: 4.808743132985067
MAE: 2.6715110628037446
Fold 6
2019-05-03 00:00:00 2021-05-27 00:00:00
2021-06-18 00:00:00 2021-08-16 00:00:00
RMSE: 1.0338743236273205
MAE: 0.8013219483003899
Fold 7
2019-07-02 00:00:00 2021-07-26 00:00:00
2021-08-17 00:00:00 2021-10-15 00:00:00
RMSE: 1.2871045814140258
MAE: 0.9206756705639029
Fold 8
2019-08-31 00:00:00 2021-09-24 00:00:00
2021-10

## + FRP

In [26]:
frp_df = train_df[['row', 'col', 'date', 'frp', 'delta_pm25_t+1']]

frp_df.head()

,row,col,date,frp,delta_pm25_t+1
0,0,0,2018-07-05,0.0,0.601407
1,0,0,2018-07-06,0.0,0.038002
2,0,0,2018-07-07,0.0,0.556278
3,0,0,2018-07-08,0.0,-0.491020
4,0,0,2018-07-09,0.0,1.041161


In [27]:
model = xgb.XGBRegressor(
        objective='reg:squarederror',
        **params,
        random_state=191
    )

rmse_scores, mae_scores = T.rolling_window_cv(
    train_days=365 * 2,
    gap_days=21,
    test_days=60,
    df=frp_df,
    model=model
)

mean_rmse = np.mean(rmse_scores)
se_rmse = np.std(rmse_scores)

mean_mae = np.mean(mae_scores)
se_mae = np.std(mae_scores)

print("Summary of Performance Statistics")
print(f"RMSE: {mean_rmse} +/- {se_rmse}")
print(f"MAE: {mean_mae} +/- {se_mae}")

Fold 1
2018-07-05 00:00:00 2020-07-19 00:00:00
2020-08-10 00:00:00 2020-10-08 00:00:00
RMSE: 1.6840806601863463
MAE: 0.9866939701518992
Fold 2
2018-09-03 00:00:00 2020-09-17 00:00:00
2020-10-09 00:00:00 2020-12-07 00:00:00
RMSE: 5.856214296988159
MAE: 2.24279982457853
Fold 3
2018-11-02 00:00:00 2020-11-16 00:00:00
2020-12-08 00:00:00 2021-02-05 00:00:00
RMSE: 7.418788635776148
MAE: 5.319242412731583
Fold 4
2019-01-01 00:00:00 2021-01-15 00:00:00
2021-02-06 00:00:00 2021-04-18 00:00:00
RMSE: 13.76411355047025
MAE: 9.424911522893476
Fold 5
2019-03-04 00:00:00 2021-03-28 00:00:00
2021-04-19 00:00:00 2021-06-17 00:00:00
RMSE: 4.77031335487193
MAE: 2.616875852536596
Fold 6
2019-05-03 00:00:00 2021-05-27 00:00:00
2021-06-18 00:00:00 2021-08-16 00:00:00
RMSE: 1.0144604436181064
MAE: 0.7788513401960787
Fold 7
2019-07-02 00:00:00 2021-07-26 00:00:00
2021-08-17 00:00:00 2021-10-15 00:00:00
RMSE: 1.2819944303115973
MAE: 0.9129499663269519
Fold 8
2019-08-31 00:00:00 2021-09-24 00:00:00
2021-10-16 

In [41]:
test_df = train_df[['row', 'col', 'date', 'elevation', 'delta_pm25_t', 'frp', 'delta_pm25_t+1']]
test_df.head()

,row,col,date,elevation,delta_pm25_t,frp,delta_pm25_t+1
0,0,0,2018-07-05,1360.0,0.302968,0.0,0.601407
1,0,0,2018-07-06,1360.0,0.601407,0.0,0.038002
2,0,0,2018-07-07,1360.0,0.038002,0.0,0.556278
3,0,0,2018-07-08,1360.0,0.556278,0.0,-0.491020
4,0,0,2018-07-09,1360.0,-0.491020,0.0,1.041161


In [42]:
model = xgb.XGBRegressor(
        objective='reg:squarederror',
        **params,
        random_state=191
    )

rmse_scores, mae_scores = T.rolling_window_cv(
    train_days=365 * 2,
    gap_days=21,
    test_days=60,
    df=test_df,
    model=model
)

mean_rmse = np.mean(rmse_scores)
se_rmse = np.std(rmse_scores)

mean_mae = np.mean(mae_scores)
se_mae = np.std(mae_scores)

print("Summary of Performance Statistics")
print(f"RMSE: {mean_rmse} +/- {se_rmse}")
print(f"MAE: {mean_mae} +/- {se_mae}")

Fold 1
2018-07-05 00:00:00 2020-07-19 00:00:00
2020-08-10 00:00:00 2020-10-08 00:00:00
RMSE: 1.6474613583700057
MAE: 0.9985698538302471
Fold 2
2018-09-03 00:00:00 2020-09-17 00:00:00
2020-10-09 00:00:00 2020-12-07 00:00:00
RMSE: 5.209639625003876
MAE: 2.1672216206463326
Fold 3
2018-11-02 00:00:00 2020-11-16 00:00:00
2020-12-08 00:00:00 2021-02-05 00:00:00
RMSE: 7.310148534575478
MAE: 5.274535988476588
Fold 4
2019-01-01 00:00:00 2021-01-15 00:00:00
2021-02-06 00:00:00 2021-04-18 00:00:00
RMSE: 13.542813070196775
MAE: 9.353974621985728
Fold 5
2019-03-04 00:00:00 2021-03-28 00:00:00
2021-04-19 00:00:00 2021-06-17 00:00:00
RMSE: 4.820305666322038
MAE: 2.6725359998821974
Fold 6
2019-05-03 00:00:00 2021-05-27 00:00:00
2021-06-18 00:00:00 2021-08-16 00:00:00
RMSE: 1.0460418126317363
MAE: 0.8127950163376567
Fold 7
2019-07-02 00:00:00 2021-07-26 00:00:00
2021-08-17 00:00:00 2021-10-15 00:00:00
RMSE: 1.2796388990269725
MAE: 0.917964265386453
Fold 8
2019-08-31 00:00:00 2021-09-24 00:00:00
2021-10